# Project 7: Book Popularity Prediction
### CSE303 Statistics for Data Science — Group Mini Project
### Predicting high- vs. low-popularity Bangla books from metadata, text, and relational features

**Dataset:** Rokomari Bangla Book Recommendation Dataset (Hugging Face: `DevnilMaster1/Bangla-Book-Recommendation-Dataset`) —
`book.json`, `author.json`, `category.json`, `publisher.json`, `review.json`, and the linking tables
(`user_to_review`, `book_to_review`, `book_to_author`, `book_to_category`, `book_to_publisher`).

**Task:** Binary classification — predict whether a book falls into the **high-popularity** or **low-popularity** group.

**Target-label rule (bestseller proxy):**
`engagement_score = rating_count + review_count`; a book is labeled **High (1)** if its engagement
score is at or above the **75th percentile** (top 25% most-engaged books) and **Low (0)** otherwise.

**No-leakage rule:** `rating_count`, `review_count`, and any column that is a direct proxy for
engagement (`average_rating`, `best_seller_badge`, `wished_customer_count`) are **excluded from the
feature set** because they define or trivially leak the label. Wherever the notebook needs the raw
`engagement_score` itself for a *statistical* analysis (correlation, ANOVA, regression), it is pulled
from the full book-level table, never from the leakage-safe modelling table used for classification.

**Techniques covered (per the course overview):** EDA · correlation · hypothesis testing · ANOVA ·
regression · classification · class-imbalance analysis · bias–variance analysis · model evaluation.


## 1. Install dependencies

In [ ]:
!pip install -q huggingface_hub tqdm scikit-learn scipy statsmodels pandas numpy matplotlib seaborn

## 2. Download the dataset from Hugging Face Hub

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

REPO_ID = "DevnilMaster1/Bangla-Book-Recommendation-Dataset"
DATA_FOLDER = "RokomariBG_Dataset"
os.makedirs(DATA_FOLDER, exist_ok=True)

FILES_NEEDED = [
    'book.json',
    'author.json',
    'category.json',
    'publisher.json',
    'review.json',
    'user_to_review.json',
    'book_to_review.json',
    'book_to_author.json',
    'book_to_category.json',
    'book_to_publisher.json'
    ]

for filename in FILES_NEEDED:
    dest = os.path.join(DATA_FOLDER, filename)
    if not os.path.exists(dest):
        try:
            downloaded_path = hf_hub_download(
                repo_id=REPO_ID,
                filename=filename,
                repo_type="dataset"
            )
            shutil.copy(downloaded_path, dest)
            print(f"Saved: {filename}")
        except Exception as e:
            print(f"Error downloading {filename}: {e}")
    else:
        print(f"Already present: {filename}")

## 3. Load the raw JSON tables into pandas DataFrames

In [ ]:
import os, json
import pandas as pd
import numpy as np

DATA_FOLDER = "RokomariBG_Dataset"

FILES_NEEDED = [
    "book.json", "author.json", "category.json", "publisher.json",
    "review.json", "user_to_review.json", "book_to_review.json",
    "book_to_author.json", "book_to_category.json", "book_to_publisher.json",
]

def load_json_table(path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except json.JSONDecodeError:
        with open(path, "r", encoding="utf-8") as f:
            data = [json.loads(line) for line in f if line.strip()]
    return pd.DataFrame(data)

tables = {}
for filename in FILES_NEEDED:
    path = os.path.join(DATA_FOLDER, filename)
    tables[filename.replace(".json", "")] = load_json_table(path)
    print(f"{filename:26s} -> {tables[filename.replace('.json','')].shape}")

book_df       = tables["book"]
author_df     = tables["author"]
category_df   = tables["category"]
publisher_df  = tables["publisher"]
review_df     = tables["review"]
u2r_df        = tables["user_to_review"]
b2r_df        = tables["book_to_review"]
b2a_df        = tables["book_to_author"]
b2c_df        = tables["book_to_category"]
b2p_df        = tables["book_to_publisher"]

## 4. Build a single book-level table
Each book is linked to (possibly several) authors, categories, and a publisher through the
`book_to_*` linking tables. We take the first-listed author/category/publisher as the book's
*primary* attribute, and also keep `n_categories` / `n_authors` as structural features
(these describe how the book is catalogued, not how popular it is, so they are safe to use).

In [ ]:
# --- primary (first-listed) author / category / publisher per book ---------
first_author = b2a_df.drop_duplicates(subset="book_id", keep="first")
first_category = b2c_df.drop_duplicates(subset="book_id", keep="first")
first_publisher = b2p_df.drop_duplicates(subset="book_id", keep="first")

# number of categories / authors a book is tagged with (structural feature,
# does not use rating/review counts, so it is safe to keep)
n_categories = b2c_df.groupby("book_id").size().rename("n_categories")
n_authors = b2a_df.groupby("book_id").size().rename("n_authors")

df = book_df.copy()
df["book_id"] = df["book_id"].astype(str)

for other, key in [(first_author, "book_id"), (first_category, "book_id"), (first_publisher, "book_id")]:
    other[key] = other[key].astype(str)

df = df.merge(first_author[["book_id", "author_id"]].rename(columns={"author_id": "author_id_link"}),
              on="book_id", how="left")
df = df.merge(first_category[["book_id", "category_id"]].rename(columns={"category_id": "category_id_link"}),
              on="book_id", how="left")
df = df.merge(first_publisher[["book_id", "publisher_id"]].rename(columns={"publisher_id": "publisher_id_link"}),
              on="book_id", how="left")

# NOTE: book_df does not necessarily already contain author_id / category_id /
# publisher_id columns (those relationships live in the book_to_* linking
# tables). Fill from the linked value when a column already exists, otherwise
# create it directly from the link -- this avoids a KeyError (fillna on a
# missing column) or an AttributeError (combine_first on a missing column).
for link_col, final_col in [("author_id_link", "author_id"),
                             ("category_id_link", "category_id"),
                             ("publisher_id_link", "publisher_id")]:
    if final_col in df.columns:
        df[final_col] = df[final_col].fillna(df[link_col])
    else:
        df[final_col] = df[link_col]

n_categories.index = n_categories.index.astype(str)
n_authors.index = n_authors.index.astype(str)
df = df.merge(n_categories, left_on="book_id", right_index=True, how="left")
df = df.merge(n_authors, left_on="book_id", right_index=True, how="left")
df["n_categories"] = df["n_categories"].fillna(0)
df["n_authors"] = df["n_authors"].fillna(0)

# --- attach author / category / publisher attributes -----------------------
author_df2 = author_df.copy()
author_df2["author_id"] = author_df2["author_id"].astype(str)
df["author_id"] = df["author_id"].astype(str)
df = df.merge(
    author_df2[["author_id", "follower_count"]].rename(columns={"follower_count": "author_follower_count"}),
    on="author_id", how="left"
)

category_df2 = category_df.copy()
category_df2["category_id"] = category_df2["category_id"].astype(str)
df["category_id"] = df["category_id"].astype(str)
df = df.merge(
    category_df2[["category_id", "category_name", "books_count"]].rename(
        columns={"books_count": "category_books_count"}),
    on="category_id", how="left"
)

publisher_df2 = publisher_df.copy()
publisher_df2["publisher_id"] = publisher_df2["publisher_id"].astype(str)
df["publisher_id"] = df["publisher_id"].astype(str)
df = df.merge(
    publisher_df2[["publisher_id", "total_authors", "total_categories"]].rename(
        columns={"total_authors": "publisher_total_authors",
                 "total_categories": "publisher_total_categories"}),
    on="publisher_id", how="left"
)

print("Book-level table shape:", df.shape)
df.head(3)

## 5. Feature engineering and target-label construction

In [ ]:
df["book_price"] = pd.to_numeric(df["book_price"], errors="coerce")
df["offer_price"] = pd.to_numeric(df["offer_price"], errors="coerce")
df["book_pages"] = pd.to_numeric(df["book_pages"], errors="coerce")
df["rating_count"] = pd.to_numeric(df["rating_count"], errors="coerce").fillna(0)
df["review_count"] = pd.to_numeric(df["review_count"], errors="coerce").fillna(0)
df["average_rating"] = pd.to_numeric(df["average_rating"], errors="coerce")

df["discount_pct"] = np.where(
    (df["book_price"] > 0) & df["offer_price"].notna(),
    (df["book_price"] - df["offer_price"]) / df["book_price"] * 100,
    np.nan
)
df["summary_length_chars"] = df["book_summary"].fillna("").apply(len)
df["has_summary"] = (df["summary_length_chars"] > 0).astype(int)
df["title_length_chars"] = df["book_title"].fillna("").apply(len)

# =============================================================================
# TARGET LABEL: "bestseller proxy" defined by an engagement percentile rule
# engagement_score = rating_count + review_count
# label = 1 (High popularity) if engagement_score >= 75th percentile
#         (i.e. book is in the top 25% most-engaged books), else 0
# =============================================================================
df["engagement_score"] = df["rating_count"] + df["review_count"]
threshold = df["engagement_score"].quantile(0.75)
df["label"] = (df["engagement_score"] >= threshold).astype(int)

print(f"75th percentile engagement threshold = {threshold:.2f}")
print(df["label"].value_counts(normalize=True))

## 6. Remove leakage columns and assemble the classification table
Columns that directly define the label (`rating_count`, `review_count`), or that are themselves
strong engagement proxies (`average_rating`, `best_seller_badge`, `wished_customer_count`), are
dropped from the feature set. Identifier columns are dropped as well (they do not generalize).

In [ ]:
# columns that directly define or trivially leak the label must be excluded
LEAKAGE_COLS = [
    "rating_count", "review_count", "engagement_score",
    "average_rating",        # rating value is an engagement-derived signal too
    "best_seller_badge",     # directly encodes popularity
    "wished_customer_count", # a popularity/engagement signal
    "stock_message", "stock_quantity", "book_availability",
    "book_url", "isbn", "book_id",
    "author_id", "author_id_link", "category_id", "category_id_link",
    "publisher_id", "publisher_id_link",
]

TEXT_COL = "book_summary"

NUMERIC_COLS = [
    "book_price", "offer_price", "discount_pct", "book_pages",
    "summary_length_chars", "has_summary", "title_length_chars",
    "n_categories", "n_authors", "author_follower_count",
    "category_books_count", "publisher_total_authors", "publisher_total_categories",
]

CATEGORICAL_COLS = ["category_name"]

model_df = df.dropna(subset=["label"]).copy()
model_df[TEXT_COL] = model_df[TEXT_COL].fillna("")
for c in NUMERIC_COLS:
    if c in model_df.columns:
        model_df[c] = pd.to_numeric(model_df[c], errors="coerce")
for c in CATEGORICAL_COLS:
    model_df[c] = model_df[c].fillna("Unknown")

keep_cols = NUMERIC_COLS + CATEGORICAL_COLS + [TEXT_COL, "label"]
model_df = model_df[keep_cols]
print("Modelling table:", model_df.shape)
print("Positive class rate:", model_df["label"].mean().round(3))

## 7. Train / validation / test split
A stratified 70% / 15% / 15% split preserves the class ratio (~25% high-popularity) in every split.

In [ ]:
from sklearn.model_selection import train_test_split

X = model_df.drop(columns=["label"])
y = model_df["label"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Train:", X_train.shape, " Val:", X_val.shape, " Test:", X_test.shape)

## 8. Preprocessing pipeline
* **Numeric** features → median imputation + standard scaling.
* **Categorical** feature (`category_name`) → most-frequent imputation + one-hot encoding (top 30 categories).
* **Text** feature (`book_summary`, Bangla) → TF-IDF (uni/bi-grams, 3000 terms) → TruncatedSVD (100 components)
  to produce a dense text embedding that can be combined with the tabular features.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", max_categories=30)),
])

text_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=3000, min_df=5, ngram_range=(1, 2))),
    ("svd", TruncatedSVD(n_components=100, random_state=42)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC_COLS),
    ("cat", categorical_pipe, CATEGORICAL_COLS),
    ("text", text_pipe, TEXT_COL),
])

## 9. Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

sns.histplot(np.log1p(df["engagement_score"]), bins=50, ax=axes[0, 0])
axes[0, 0].axvline(np.log1p(threshold), color="red", linestyle="--", label="75th pct threshold")
axes[0, 0].set_title("Fig. 1a: log(1+engagement_score) distribution")
axes[0, 0].legend()

sns.boxplot(x="label", y="book_price", data=df[df["book_price"] < df["book_price"].quantile(0.99)], ax=axes[0, 1])
axes[0, 1].set_title("Fig. 1b: Price by popularity label")

sns.countplot(x="label", data=df, ax=axes[1, 0])
axes[1, 0].set_title("Fig. 1c: Class balance (0=Low, 1=High)")

top_cats = df["category_name"].value_counts().head(10)
sns.barplot(x=top_cats.values, y=top_cats.index, ax=axes[1, 1])
axes[1, 1].set_title("Fig. 1d: Top-10 categories by book count")

plt.tight_layout()
plt.savefig("fig1_eda_overview.png", dpi=150)
plt.show()

print(df[["book_price", "book_pages", "average_rating", "discount_pct",
          "author_follower_count", "engagement_score"]].describe())

## 10. Correlation analysis
Before modelling, we check how strongly each candidate numeric feature relates to the
**continuous** `engagement_score` (not the binary label, and not `rating_count`/`review_count`
themselves). We report both **Pearson** (linear) and **Spearman** (monotonic, rank-based) correlation,
since engagement counts are heavily right-skewed and Spearman is more robust to that skew.

In [ ]:
from scipy.stats import pearsonr, spearmanr

corr_features = [
    "book_price", "offer_price", "discount_pct", "book_pages",
    "summary_length_chars", "title_length_chars", "n_categories", "n_authors",
    "author_follower_count", "category_books_count",
    "publisher_total_authors", "publisher_total_categories",
]

corr_rows = []
for feat in corr_features:
    sub = df[[feat, "engagement_score"]].dropna()
    if len(sub) < 3:
        continue
    r_p, p_p = pearsonr(sub[feat], sub["engagement_score"])
    r_s, p_s = spearmanr(sub[feat], sub["engagement_score"])
    corr_rows.append({"feature": feat, "n": len(sub),
                       "pearson_r": r_p, "pearson_p": p_p,
                       "spearman_r": r_s, "spearman_p": p_s})

corr_table = pd.DataFrame(corr_rows).sort_values("spearman_r", key=abs, ascending=False)
print(corr_table.round(4).to_string(index=False))

# Correlation is computed against engagement_score (the pre-threshold continuous
# quantity), NOT against the binary label, and NOT including rating_count /
# review_count themselves, so this analysis does not touch the leakage columns.

plt.figure(figsize=(7, 6))
heat_cols = corr_features + ["engagement_score"]
sns.heatmap(df[heat_cols].corr(method="spearman"), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": 0.8})
plt.title("Fig. 3: Spearman correlation heatmap (features vs. engagement_score)")
plt.tight_layout()
plt.savefig("fig3_correlation_heatmap.png", dpi=150)
plt.show()

## 11. Hypothesis testing
Two independent hypothesis tests, each with an explicit null hypothesis, test statistic, and
decision at α = 0.05:

* **H1 (price vs. popularity):** is there a difference in `book_price` between High- and
  Low-popularity books? We check normality with Shapiro-Wilk first; because price is not expected
  to be normally distributed, the non-parametric **Mann-Whitney U** test is the primary result,
  with Welch's t-test reported alongside for reference.
* **H2 (category vs. popularity):** is popularity tier independent of book category? Tested with a
  **chi-square test of independence** on a category × label contingency table (top-10 categories).

In [ ]:
from scipy.stats import mannwhitneyu, ttest_ind, chi2_contingency, shapiro

# -----------------------------------------------------------------------
# H1: Does mean/median PRICE differ between High- and Low-popularity books?
#     H0: no difference in price between the two popularity groups
# -----------------------------------------------------------------------
price_high = model_df.loc[model_df["label"] == 1, "book_price"].dropna()
price_low  = model_df.loc[model_df["label"] == 0, "book_price"].dropna()

# Normality check (Shapiro on a capped sample, since Shapiro is unreliable for very large n)
sample_n = min(500, len(price_high), len(price_low))
sh_high = shapiro(price_high.sample(sample_n, random_state=42))
sh_low  = shapiro(price_low.sample(sample_n, random_state=42))
print(f"Shapiro-Wilk (High-popularity price sample): W={sh_high.statistic:.3f}, p={sh_high.pvalue:.4g}")
print(f"Shapiro-Wilk (Low-popularity price sample):  W={sh_low.statistic:.3f}, p={sh_low.pvalue:.4g}")
# Price is typically not normally distributed at this scale, so we report the
# non-parametric Mann-Whitney U test as the primary result, alongside Welch's
# t-test for reference.

u_stat, u_p = mannwhitneyu(price_high, price_low, alternative="two-sided")
t_stat, t_p = ttest_ind(price_high, price_low, equal_var=False)  # Welch's t-test

print(f"\nMann-Whitney U test (price, High vs Low popularity):")
print(f"  U = {u_stat:.1f}, p-value = {u_p:.4g}")
print(f"Welch's t-test (price, High vs Low popularity):")
print(f"  t = {t_stat:.3f}, p-value = {t_p:.4g}")
print(f"  Mean price - High: {price_high.mean():.2f} BDT, Low: {price_low.mean():.2f} BDT")
print(f"  Median price - High: {price_high.median():.2f} BDT, Low: {price_low.median():.2f} BDT")

alpha = 0.05
conclusion_price = "reject H0 (significant difference)" if u_p < alpha else "fail to reject H0 (no significant difference)"
print(f"  Conclusion at alpha={alpha}: {conclusion_price}")

# -----------------------------------------------------------------------
# H2: Is CATEGORY independent of popularity tier?
#     H0: category and popularity label are independent
# -----------------------------------------------------------------------
top_categories = model_df["category_name"].value_counts().head(10).index
sub_cat = model_df[model_df["category_name"].isin(top_categories)]
contingency = pd.crosstab(sub_cat["category_name"], sub_cat["label"])

chi2_stat, chi2_p, dof, expected = chi2_contingency(contingency)
print(f"\nChi-square test of independence (category x popularity label, top-10 categories):")
print(f"  chi2 = {chi2_stat:.3f}, dof = {dof}, p-value = {chi2_p:.4g}")
conclusion_cat = "reject H0 (category and popularity are associated)" if chi2_p < alpha else "fail to reject H0 (no evidence of association)"
print(f"  Conclusion at alpha={alpha}: {conclusion_cat}")
print(contingency)

## 12. ANOVA — does mean engagement differ across categories?
A **one-way ANOVA** tests H0: mean `engagement_score` is equal across the top-10 categories.
If the ANOVA is significant, a **Tukey HSD** post-hoc test identifies which category pairs differ.

In [ ]:
from scipy.stats import f_oneway
import scipy.stats as st

# H0: mean engagement_score is equal across the top-N categories
# engagement_score was dropped from model_df (it is leakage for the classifier),
# so we pull it back from the full book-level table `df` for this ANOVA only.
anova_df = df[df["category_name"].isin(top_categories)][["category_name", "engagement_score"]].dropna()
groups = [g["engagement_score"].values for _, g in anova_df.groupby("category_name")]

f_stat, anova_p = f_oneway(*groups)
print(f"One-way ANOVA (engagement_score across top-10 categories):")
print(f"  F = {f_stat:.3f}, p-value = {anova_p:.4g}")
conclusion_anova = "reject H0 (at least one category has a different mean engagement)" \
    if anova_p < alpha else "fail to reject H0 (no evidence that mean engagement differs by category)"
print(f"  Conclusion at alpha={alpha}: {conclusion_anova}")

anova_summary = anova_df.groupby("category_name")["engagement_score"].agg(["mean", "std", "count"]).round(2)
print("\nGroup summary:")
print(anova_summary)

# Post-hoc: Tukey HSD (only meaningful / run if ANOVA is significant)
if anova_p < alpha:
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    tukey = pairwise_tukeyhsd(anova_df["engagement_score"], anova_df["category_name"], alpha=alpha)
    print("\nTukey HSD post-hoc test:")
    print(tukey.summary())

plt.figure(figsize=(9, 5))
sns.boxplot(x="category_name", y="engagement_score", data=anova_df, showfliers=False)
plt.xticks(rotation=45, ha="right")
plt.title("Fig. 4: engagement_score by category (top-10, outliers hidden)")
plt.tight_layout()
plt.savefig("fig4_anova_boxplot.png", dpi=150)
plt.show()

## 13. Regression — predicting the continuous engagement score
This section treats **regression as its own technique**, separate from classification: we fit a
Linear Regression model to predict `log1p(engagement_score)` (continuous) directly from the
leakage-safe feature set, report R²/MAE/RMSE, inspect standardized coefficients, and check
residual diagnostics (residuals-vs-fitted, Q-Q plot) for the linearity/homoscedasticity/normality
assumptions behind the model.

In [ ]:
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Regression target: log1p(engagement_score) -- a continuous, pre-threshold
# quantity. This is a DIFFERENT task from the classification above: here we
# predict the (transformed) count itself, not the binary label, so this
# section exercises "regression" as its own technique, not just as a
# classifier backbone.
reg_df = df.dropna(subset=["engagement_score"]).copy()
reg_df[TEXT_COL] = reg_df[TEXT_COL].fillna("")
for c in NUMERIC_COLS:
    reg_df[c] = pd.to_numeric(reg_df[c], errors="coerce")
reg_df["category_name"] = reg_df["category_name"].fillna("Unknown")
reg_df["y_log_engagement"] = np.log1p(reg_df["engagement_score"])

X_reg = reg_df[NUMERIC_COLS + CATEGORICAL_COLS + [TEXT_COL]]
y_reg = reg_df["y_log_engagement"]

Xr_train, Xr_temp, yr_train, yr_temp = train_test_split(X_reg, y_reg, test_size=0.30, random_state=42)
Xr_val, Xr_test, yr_val, yr_test = train_test_split(Xr_temp, yr_temp, test_size=0.50, random_state=42)

from sklearn.linear_model import LinearRegression
reg_pipe = Pipeline([("prep", preprocessor), ("reg", LinearRegression())])
reg_pipe.fit(Xr_train, yr_train)

pred_test = reg_pipe.predict(Xr_test)
r2 = r2_score(yr_test, pred_test)
mae = mean_absolute_error(yr_test, pred_test)
rmse = mean_squared_error(yr_test, pred_test) ** 0.5
print(f"Linear Regression on log1p(engagement_score) -- test set:")
print(f"  R^2   = {r2:.4f}")
print(f"  MAE   = {mae:.4f}")
print(f"  RMSE  = {rmse:.4f}")

# Coefficient table for the numeric block only (interpretable, standardized units)
num_coefs = reg_pipe.named_steps["reg"].coef_[:len(NUMERIC_COLS)]
coef_table = pd.DataFrame({"feature": NUMERIC_COLS, "standardized_coef": num_coefs}) \
    .sort_values("standardized_coef", key=abs, ascending=False)
print("\nStandardized numeric coefficients (log-engagement target):")
print(coef_table.round(4).to_string(index=False))

# Residual diagnostics
residuals = yr_test.values - pred_test
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(pred_test, residuals, alpha=0.3, s=10)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Predicted log1p(engagement_score)")
axes[0].set_ylabel("Residual")
axes[0].set_title("Fig. 5a: Residuals vs. fitted")

sm.qqplot(residuals, line="s", ax=axes[1])
axes[1].set_title("Fig. 5b: Normal Q-Q plot of residuals")
plt.tight_layout()
plt.savefig("fig5_regression_diagnostics.png", dpi=150)
plt.show()

## 14. Baseline models
1. **Majority-class baseline** — always predicts the majority (low-popularity) class.
2. **Logistic Regression** — a standard, interpretable linear baseline with class-balanced weights.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)

def evaluate(name, pipe, X_tr, y_tr, X_ev, y_ev, results):
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_ev)
    proba = pipe.predict_proba(X_ev)[:, 1] if hasattr(pipe, "predict_proba") else pred
    row = {
        "model": name,
        "accuracy": accuracy_score(y_ev, pred),
        "precision": precision_score(y_ev, pred, zero_division=0),
        "recall": recall_score(y_ev, pred, zero_division=0),
        "f1": f1_score(y_ev, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_ev, proba),
    }
    results.append(row)
    print(f"--- {name} ---")
    print(classification_report(y_ev, pred, digits=3))
    return pipe, row

results = []

baseline_majority = Pipeline([("prep", preprocessor),
                               ("clf", DummyClassifier(strategy="most_frequent"))])
_, r1 = evaluate("Baseline: Majority class", baseline_majority, X_train, y_train, X_val, y_val, results)

baseline_lr = Pipeline([("prep", preprocessor),
                         ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
lr_fitted, r2 = evaluate("Baseline: Logistic Regression", baseline_lr, X_train, y_train, X_val, y_val, results)

## 15. Proposed model
A **Gradient Boosting Classifier** trained on the combined tabular + text-embedding feature space,
tuned with `RandomizedSearchCV` (3-fold CV, optimizing F1) over tree depth, learning rate,
number of estimators, and subsample ratio.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV

proposed_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf", GradientBoostingClassifier(random_state=42)),
])

param_dist = {
    "clf__n_estimators": [100, 200, 300],
    "clf__max_depth": [2, 3, 4],
    "clf__learning_rate": [0.03, 0.05, 0.1],
    "clf__subsample": [0.7, 0.85, 1.0],
}

search = RandomizedSearchCV(
    proposed_pipe, param_dist, n_iter=10, scoring="f1",
    cv=3, random_state=42, n_jobs=-1
)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)

best_model = search.best_estimator_
_, r3 = evaluate("Proposed: Gradient Boosting (tabular+text)", best_model, X_train, y_train, X_val, y_val, results)

## 16. Final evaluation on the held-out test set

In [ ]:
final_results = []
for name, pipe in [("Baseline: Majority class", baseline_majority),
                    ("Baseline: Logistic Regression", lr_fitted),
                    ("Proposed: Gradient Boosting", best_model)]:
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else pred
    final_results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, proba),
    })

results_df = pd.DataFrame(final_results)
print(results_df.round(3))
results_df.to_csv("test_set_results.csv", index=False)

cm = confusion_matrix(y_test, best_model.predict(X_test))
plt.figure(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Low", "High"], yticklabels=["Low", "High"])
plt.title("Fig. 2: Confusion matrix (test set, proposed model)")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.savefig("fig2_confusion_matrix.png", dpi=150)
plt.show()

## 17. Class-imbalance analysis
The label is imbalanced (~75% Low / 25% High). We quantify the imbalance ratio and directly compare
an **unbalanced** vs. **class-weight-balanced** Logistic Regression to make the practical effect of
imbalance concrete: expect the unbalanced model to look better on raw accuracy while doing worse on
minority-class (High-popularity) recall.

In [ ]:
imbalance_ratio = (y_train.value_counts(normalize=True))
print("Class distribution (train set):")
print(imbalance_ratio.round(4))
print(f"Imbalance ratio (majority:minority) = {imbalance_ratio.max() / imbalance_ratio.min():.2f} : 1")

# Compare an UNBALANCED vs BALANCED logistic regression to make the effect of
# class imbalance concrete, rather than only asserting it matters.
lr_unbalanced = Pipeline([("prep", preprocessor), ("clf", LogisticRegression(max_iter=1000))])
lr_balanced   = Pipeline([("prep", preprocessor), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])

imbalance_rows = []
for name, pipe in [("Logistic Regression (unbalanced)", lr_unbalanced),
                    ("Logistic Regression (class_weight=balanced)", lr_balanced)]:
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_val)
    proba = pipe.predict_proba(X_val)[:, 1]
    imbalance_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_val, pred),
        "precision_high": precision_score(y_val, pred, zero_division=0),
        "recall_high": recall_score(y_val, pred, zero_division=0),
        "f1_high": f1_score(y_val, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_val, proba),
    })

imbalance_table = pd.DataFrame(imbalance_rows)
print("\nEffect of class-weight balancing (validation set):")
print(imbalance_table.round(3).to_string(index=False))
# Expect: unbalanced model favors the majority (Low) class -> higher accuracy but
# lower recall on the High-popularity class; balancing trades some accuracy for
# substantially better recall on the minority class.

## 18. Bias–variance analysis
* **Validation curve:** vary tree `max_depth` (model complexity) and compare training vs. validation
  F1. A small train/val gap at low depth with low absolute scores signals **high bias**
  (underfitting); a growing gap as depth increases signals **high variance** (overfitting).
* **Learning curve:** vary training-set size and compare training vs. validation F1, to see whether
  more data would be expected to help (still converging) or not (already plateaued).

In [ ]:
from sklearn.model_selection import validation_curve, learning_curve
from sklearn.ensemble import GradientBoostingClassifier

simple_pipe = Pipeline([("prep", preprocessor),
                         ("clf", GradientBoostingClassifier(n_estimators=150, random_state=42))])

# --- Validation curve: model complexity (max_depth) vs. train/val score -----
depth_range = [1, 2, 3, 4, 5, 6, 8]
train_scores, val_scores = validation_curve(
    simple_pipe, X_train, y_train,
    param_name="clf__max_depth", param_range=depth_range,
    scoring="f1", cv=3, n_jobs=-1
)

plt.figure(figsize=(7, 5))
plt.plot(depth_range, train_scores.mean(axis=1), "o-", label="Training F1")
plt.plot(depth_range, val_scores.mean(axis=1), "o-", label="Validation F1")
plt.fill_between(depth_range, train_scores.mean(axis=1) - train_scores.std(axis=1),
                  train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.15)
plt.fill_between(depth_range, val_scores.mean(axis=1) - val_scores.std(axis=1),
                  val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.15)
plt.xlabel("Tree max_depth (model complexity)")
plt.ylabel("F1 score")
plt.title("Fig. 6: Validation curve -- bias/variance vs. tree depth")
plt.legend()
plt.tight_layout()
plt.savefig("fig6_validation_curve.png", dpi=150)
plt.show()

gap = train_scores.mean(axis=1) - val_scores.mean(axis=1)
print("max_depth : train_F1 : val_F1 : gap(train-val)")
for d, tr, va, g in zip(depth_range, train_scores.mean(axis=1), val_scores.mean(axis=1), gap):
    print(f"  {d:>3}     :  {tr:.3f}  : {va:.3f}  : {g:+.3f}")
# A small, roughly-constant gap and low absolute scores at shallow depth signal
# high bias (underfitting). A growing gap (train >> val) at large depth signals
# high variance (overfitting). The depth with the highest validation F1, not the
# highest training F1, is the bias-variance-appropriate choice.

# --- Learning curve: training-set size vs. train/val score ------------------
train_sizes, lc_train_scores, lc_val_scores = learning_curve(
    simple_pipe, X_train, y_train, cv=3, scoring="f1",
    train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=-1, random_state=42
)

plt.figure(figsize=(7, 5))
plt.plot(train_sizes, lc_train_scores.mean(axis=1), "o-", label="Training F1")
plt.plot(train_sizes, lc_val_scores.mean(axis=1), "o-", label="Validation F1")
plt.xlabel("Training set size")
plt.ylabel("F1 score")
plt.title("Fig. 7: Learning curve (proposed model family)")
plt.legend()
plt.tight_layout()
plt.savefig("fig7_learning_curve.png", dpi=150)
plt.show()

## 19. Notes on the "early popularity prediction" extension
`review.json` contains `review_date` for every review, and `book_to_review.json` links reviews to
books. A natural extension of this project is **early popularity prediction**: for each book, take
only the reviews posted within the first *N* days after its first review, compute an
early-engagement count from those, and use *that* (instead of the full-history `engagement_score`)
to build the label — while restricting every input feature to information that would have been
available by that same cutoff date. We describe this extension here as future work; the current
notebook uses the full-history engagement rule for the primary experiment because it keeps the
label definition simple and auditable given the available fields.